# Diagnose App Startup Crash
This notebook is for reproducing the Android app startup failure, capturing logs, identifying the cause, and verifying the fix.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

print("Diagnostic libraries imported")

## Reproduce the Crash
Run the app on the emulator and capture whether the launch succeeds or fails.

In [ ]:
sdk_path = Path(r"C:/Users/jross/AppData/Local/Android/Sdk")
adb_path = sdk_path / "platform-tools" / "adb.exe"
device = "emulator-5554"
app_package = "com.example.eggplantdetector"
activity = ".MainActivity"

def run_cmd(cmd):
    print(f"Running: {cmd}")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print("returncode:", result.returncode)
    print("stdout:\n", result.stdout.strip())
    print("stderr:\n", result.stderr.strip())
    return result

run_cmd(f'"{adb_path}" -s {device} shell am force-stop {app_package}')
run_cmd(f'"{adb_path}" -s {device} logcat -c')
run_cmd(f'"{adb_path}" -s {device} shell am start -W -n {app_package}/{activity}')

## Capture Error Logs
Collect the error output from logcat after the app launch.

In [ ]:
logcat_cmd = f'"{adb_path}" -s {device} logcat -d'
result = subprocess.run(logcat_cmd, shell=True, capture_output=True, text=True)
error_lines = [
    line for line in result.stdout.splitlines()
    if any(token in line for token in ["FATAL EXCEPTION", "AndroidRuntime", app_package, "Exception", "Caused by"])
]
print("Captured log lines:")
for line in error_lines[-50:]:
    print(line)

## Analyze the Traceback
Parse the collected logs to identify the failing file, function, or missing resource.

In [ ]:
for line in error_lines:
    if "Caused by" in line or "FATAL EXCEPTION" in line or "java." in line or "kotlin." in line:
        print(line)

if not error_lines:
    print("No relevant error lines found; verify that logcat captured the crash.")

## Check Dependencies and Environment
Inspect the APK contents and assets to ensure required model files are present.

In [ ]:
apk_path = Path(r"c:/eggplant70-main-main/eggplant70-main-main/app/build/outputs/apk/debug/app-debug.apk")
if apk_path.exists():
    import zipfile
    with zipfile.ZipFile(apk_path) as z:
        for name in sorted(z.namelist()):
            if name.startswith("assets/") and ("model" in name or name.endswith(".tflite") or name.endswith("labels.txt")):
                print(name)
else:
    print("APK not found:", apk_path)

## Fix and Validate
Apply the likely fix in source and rerun the app to verify it no longer closes immediately.

In [ ]:
print("Edit source code to fix missing model asset or fallback if needed, then rerun the reproduction cell.")